# ¿Cuántos símbolos hacen falta para decidir?
## Monitoreo secuencial de traslape espectral en señales ópticas IQ: features espectrales del error, posterior calibrado y parada óptima

**Escenario:** 32 GBd, back-to-back (`32G_B2B`) · **Decisión:** ¿el canal vecino se traslapa con el canal de interés (z = 1) o no (z = 0)?

### Problema
Se observan bloques consecutivos de símbolos IQ recibidos. Tras cada bloque se puede **decidir** (declarar traslape o no) o **seguir observando**. Observar cuesta; equivocarse también, y los dos errores no cuestan igual (un falso negativo es más caro que un falso positivo). La pregunta es cuántos símbolos hay que mirar antes de decidir.

### Ruta del notebook
| Sección | Qué hace |
|---|---|
| 1. Entorno y parámetros | Configuración única; todo lo ajustable está aquí |
| 2. Datos y particiones | Inventario, condiciones compartidas, roles de cada canción |
| 3. Señal y features | Vector de error, EVM y autocorrelación del error: qué separa las clases |
| 4. Posterior secuencial | Features acumuladas por bloque → creencia calibrada `b_k = P(z=1 \| datos hasta k)` |
| 5. Valor de observar | Costo de decidir tras N bloques, equilibrio con el costo de observación |
| 6. Políticas de parada | Trivial, ventana fija, umbral, SARSA, Q-learning y DP, con barrido de `C_OBS` |
| 7. Sensibilidad | Comparación entre configuraciones guardadas (horizonte, nº de ventanas) |
| 8. Conclusiones | Lo que los resultados sí muestran, límites y trabajo pendiente |
| Apéndice A | Resultado negativo: el enfoque KDE + temperatura |

### Cómo usarlo
Ejecutar de arriba hacia abajo. Para repetir con otro horizonte o más ventanas por captura, cambiar `MAX_BLOCKS` o `N_SLICES_PER_CAPTURE` en la sección 1 y volver a ejecutar **todo**: cada corrida guarda sus resultados con una etiqueta (`H10_S5`, `H30_S5`, ...) y la sección 7 los compara.

> **Higiene experimental.** song1 ajusta el posterior, song2 entrena las políticas de RL, song3 sirve para elegir hiperparámetros y referencias, y **song4 solo se evalúa al final**.

---
## 1. Entorno y parámetros

In [ ]:
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from scipy.stats import kurtosis, rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (p for p in candidates if (p / "src").is_dir() and (p / "configs/experiment.yaml").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("No encontré la raíz del proyecto (debe tener src/ y configs/experiment.yaml).")
sys.path.insert(0, str(ROOT))

from src.data.inventory import InventoryConfig, build_inventory
from src.data.loader import load_capture

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)

In [ ]:
with open(ROOT / "configs/experiment.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

RS_GBAUD = float(cfg["data"]["rs_gbaud"])
ROLLOFF = float(cfg["data"]["rolloff"])
BOUNDARY_GHZ = RS_GBAUD * (1.0 + ROLLOFF)          # frontera geométrica: z = 1 si ChSp < BOUNDARY_GHZ

SEED = 7
np.random.seed(SEED)

# ---- Datos ----
FOCUS_CHSP = [28.5, 29.0, 29.5, 30.0, 30.5, 31.0, 31.5, 32.0, 32.5, 33.0, 33.5, 34.0, 34.5, 35.0, 35.5, 36.0, 37.5]

# ---- Episodios: cada episodio son MAX_BLOCKS bloques de BLOCK_SIZE símbolos ----
BLOCK_SIZE = 100
MAX_BLOCKS = 10                 # horizonte H (probar 30 para la sección 7)
N_SLICES_PER_CAPTURE = 5        # episodios por captura (probar 30 para la sección 7)
RUN_TAG = f"H{MAX_BLOCKS}_S{N_SLICES_PER_CAPTURE}"

# ---- Costos (ilustrativos, ver sección 5) ----
C_FN = 10.0                     # declarar "sin traslape" cuando sí hay
C_FP = 5.0                      # declarar "traslape" cuando no hay
BAYES_THR = C_FP / (C_FP + C_FN)  # declarar traslape si b >= 1/3 (óptimo si b está calibrada)
C_OBS_SWEEP = [0.005, 0.01, 0.02, 0.05]   # costo por bloque observado: se barre, no se fija

# ---- Posterior ----
FINAL_VARIANT = "C complejo 1..8 (re+im)"
C_REG = 0.3                     # regularización de la regresión logística

# ---- Aprendizaje por refuerzo tabular ----
SEEDS = [7, 42, 101, 202, 303]
N_BINS, ALPHA, N_EP = 31, 0.1, 5000
EPS0, EPS1 = 0.3, 0.02

print(f"Frontera geométrica: {BOUNDARY_GHZ:.2f} GHz | configuración: {RUN_TAG} "
      f"({MAX_BLOCKS * BLOCK_SIZE} símbolos por episodio)")

---
## 2. Datos y particiones

Se usan las condiciones (ChSp, OSNR) presentes en **las cuatro canciones**, de modo que cada partición vea el mismo conjunto de condiciones con contenido distinto. La etiqueta es geométrica: z = 1 si ChSp < 35.2 GHz.

| Partición | Canción | Rol |
|---|---|---|
| `df_post` | song1 | ajustar el posterior (un modelo por número de bloques) |
| `df_policy` | song2 | entrenar SARSA, Q-learning y DP |
| `df_val` | song3 | elegir hiperparámetros, ventana fija y umbrales |
| `df_test` | song4 | evaluación final, una sola vez |

In [ ]:
inventory_cfg = InventoryConfig.from_yaml(ROOT / "configs/experiment.yaml")
inv = build_inventory(inventory_cfg)
inv.to_csv(PROCESSED_DIR / "inventory.csv", index=False)


def song_from_capture_id(capture_id):
    name = str(capture_id).rsplit("/", 1)[-1]
    return name.split("_", 1)[0]


inv["song"] = inv["capture_id"].map(song_from_capture_id)

focus_inv = inv[
    inv["scenario"].eq("32G_B2B")
    & inv["chsp_ghz"].isin(FOCUS_CHSP)
    & inv["label_geometrica"].notna()
].copy()

songs = ["song1", "song2", "song3", "song4"]
condition_sets = {
    s: set(zip(focus_inv.loc[focus_inv["song"].eq(s), "chsp_ghz"].astype(float),
               focus_inv.loc[focus_inv["song"].eq(s), "osnr_db"].astype(float).round(2)))
    for s in songs
}
common_conditions = set.intersection(*condition_sets.values())
focus_inv = focus_inv[
    focus_inv.apply(lambda r: (float(r["chsp_ghz"]), round(float(r["osnr_db"]), 2)) in common_conditions, axis=1)
].copy()

print(f"Condiciones (ChSp, OSNR) compartidas por las 4 canciones: {len(common_conditions)}")
display(focus_inv.groupby(["song", "label_geometrica"]).size().unstack(fill_value=0))

In [ ]:
df_post = focus_inv[focus_inv["song"].eq("song1")].copy()
df_policy = focus_inv[focus_inv["song"].eq("song2")].copy()
df_val = focus_inv[focus_inv["song"].eq("song3")].copy()
df_test = focus_inv[focus_inv["song"].eq("song4")].copy()

PARTITIONS = {"post": df_post, "policy": df_policy, "val": df_val, "test": df_test}
summary = pd.DataFrame({
    name: {"capturas": len(df), "z=1": int(df["label_geometrica"].sum()),
           "z=0": int((df["label_geometrica"] == 0).sum()),
           "tasa base z=1": round(float(df["label_geometrica"].mean()), 3)}
    for name, df in PARTITIONS.items()
}).T
display(summary)
assert all(len(df) == len(common_conditions) for df in PARTITIONS.values())

---
## 3. Señal y features: qué separa las clases

Para un bloque de símbolos recibidos $y_t$ (16-QAM normalizada), el **vector de error** es $e_t = y_t - \hat{s}_t$, con $\hat{s}_t$ el símbolo de la constelación más cercano. A partir de él se definen dos familias de features:

- **EVM²** $=\frac{1}{n}\sum_t |e_t|^2$: mide cuánta energía de error hay, pero no de dónde viene. Lo domina el OSNR.
- **Autocorrelación compleja normalizada** $\rho_\ell=\dfrac{\sum_t e_t\,\overline{e_{t+\ell}}}{\sum_t |e_t|^2}$: el ruido ASE es blanco ($\rho_\ell\approx 0$), mientras que la interferencia de un canal vecino de banda limitada deja estructura en el error. Que esa estructura exista es una **hipótesis** que esta sección contrasta con los datos.

### 3.1 Constelación y vector de error

In [ ]:
grid_1d = np.array([-3.0, -1.0, 1.0, 3.0]) / np.sqrt(10.0)
constellation = np.array([i + 1j * q for i in grid_1d for q in grid_1d])


def error_vector(y):
    """Error complejo respecto al símbolo 16-QAM más cercano (decisión dura)."""
    idx = np.argmin(np.abs(y[:, None] - constellation[None, :]), axis=1)
    return y - constellation[idx]


example_row = focus_inv[focus_inv["label_geometrica"] == 1].iloc[0]
y_show = load_capture(example_row).y[:5000]
e_show = error_vector(y_show)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].scatter(y_show.real, y_show.imag, s=3, alpha=0.35)
axes[0].scatter(constellation.real, constellation.imag, c="red", marker="x", s=60, label="símbolos ideales")
axes[0].set(title=f"IQ recibido — ChSp={example_row['chsp_ghz']} GHz, OSNR={example_row['osnr_db']} dB",
            xlabel="I", ylabel="Q")
axes[0].legend()
axes[1].scatter(e_show.real, e_show.imag, s=3, alpha=0.35, color="tab:green")
axes[1].set(title="Vector de error $e_t$", xlabel="Re(e)", ylabel="Im(e)")
for ax in axes:
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Diagnóstico por captura: ¿qué feature separa y es invariante al OSNR?

Se calculan, para cada captura del escenario (incluidas las de ChSp muy grande, que sirven de referencia sin traslape), el EVM², la autocorrelación del error en los lags 1, 2, 3 y 5 y la curtosis de I y Q. El **residuo** de EVM² es la parte del EVM² que no explica el OSNR (se ajusta con las capturas de ChSp ≥ 36 GHz); es una referencia física que **no se puede usar en despliegue** porque requiere conocer el OSNR.

La primera ejecución recorre todas las capturas y puede tardar varios minutos; el resultado se guarda en `data/processed/features_por_captura.csv`.

In [ ]:
LAGS_DIAG = (1, 2, 3, 5)


def capture_features(y):
    e = error_vector(y)
    p = np.sum(np.abs(e) ** 2)
    f = {"evm2": p / len(e)}
    for lag in LAGS_DIAG:
        f[f"acf{lag}"] = np.real(np.sum(e[:-lag] * np.conj(e[lag:]))) / p
    f["kurt_I"] = kurtosis(e.real)
    f["kurt_Q"] = kurtosis(e.imag)
    return f


def auc(score, label):
    """AUC por rangos (Mann-Whitney). 0.5 = azar; <0.5 = la feature va en sentido contrario."""
    score, label = np.asarray(score), np.asarray(label)
    n1, n0 = (label == 1).sum(), (label == 0).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(score)
    return (r[label == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


FEAT_CACHE = PROCESSED_DIR / "features_por_captura.csv"
if FEAT_CACHE.exists():
    feat = pd.read_csv(FEAT_CACHE)
    print(f"Cargado desde caché: {FEAT_CACHE.name} ({len(feat)} capturas)")
else:
    inv_all = inv[inv["scenario"].eq("32G_B2B")].copy()
    rows = []
    for _, r in inv_all.iterrows():
        rows.append({"song": r["song"], "chsp": float(r["chsp_ghz"]), "osnr": float(r["osnr_db"]),
                     "z": int(r["label_geometrica"]), **capture_features(load_capture(r).y)})
    feat = pd.DataFrame(rows)
    feat.to_csv(FEAT_CACHE, index=False)
    print(f"Calculado y guardado: {FEAT_CACHE.name} ({len(feat)} capturas)")

feat["noise"] = 10 ** (-feat["osnr"] / 10)
ref = feat[feat["chsp"] >= 36]
slope, intercept = np.polyfit(ref["noise"], ref["evm2"], 1)
feat["resid"] = feat["evm2"] - (slope * feat["noise"] + intercept)

cols = ["resid", "acf1", "acf2", "acf3", "acf5", "kurt_I", "kurt_Q"]
print("Media de cada feature por ChSp (con pocas capturas en ChSp ≤ 29.5, interpretar con cautela):")
display(feat.groupby("chsp")[cols].mean().round(4))

In [ ]:
feats_to_test = ["evm2", "resid", "acf1", "acf2", "acf3", "acf5", "kurt_I", "kurt_Q"]
groups = {
    "Detectable: ChSp<=33 vs z=0": feat[(feat["chsp"] <= 33.0) | (feat["z"] == 0)],
    "Frontera: 34-35 vs z=0": feat[feat["chsp"].between(34.0, 35.0) | (feat["z"] == 0)],
    "Etiqueta completa (z=1 vs z=0)": feat,
}
auc_tab = pd.DataFrame({name: {f: auc(df[f], df["z"]) for f in feats_to_test} for name, df in groups.items()})
print("AUC por feature (0.5 = azar; valores < 0.5 indican que el traslape la vuelve MÁS NEGATIVA):")
display(auc_tab.round(3))

z0 = feat[feat["z"] == 0]
print("\nCorrelación con OSNR dentro de z=0 (cercana a 0 = invariante al ruido):")
print(z0[feats_to_test].corrwith(z0["osnr"]).round(3).to_string())

tmp = feat.assign(grupo=np.where(feat["chsp"] <= 32, "traslape fuerte (<=32)",
                                 np.where(feat["chsp"] >= 35.5, "sin traslape (>=35.5)", "intermedio")))
print("\nacf1 medio por canción (¿estable entre canciones?):")
display(tmp.pivot_table(index="grupo", columns="song", values="acf1", aggfunc="mean").round(4))

**Lectura (corrida de referencia).**

- El EVM² crudo separa mal (AUC 0.69 con la etiqueta completa) y va casi todo por el OSNR (correlación −0.92 dentro de z=0). La curtosis tampoco sirve: correlaciona 0.93 con el OSNR, o sea que mide ruido.
- `acf1` es la feature más limpia: correlación con OSNR de −0.03, y el traslape la vuelve **más negativa** (por eso su AUC sale por debajo de 0.5; el poder de separación es 1 − AUC: ≈ 0.87 en "detectable", 0.71 en la frontera y 0.80 con la etiqueta completa).
- Con traslape fuerte el patrón alterna de signo entre lags (`acf1` < 0, `acf2` > 0, `acf3` < 0): una firma oscilatoria compatible con energía cerca del borde de la banda. Es una interpretación, no está verificada.
- `acf1` es estable entre canciones (diferencia ≈ 0.01 frente a una brecha entre clases de ≈ 0.14), así que el desajuste entre canciones **no** es el problema.
- La frontera física que ve `acf1` cae entre 34.5 y 35.0 GHz, cerca de la frontera geométrica de 35.2 GHz; el residuo de EVM² la veía más abajo porque es un instrumento menos sensible.

---
## 4. Posterior secuencial calibrado

### Idea
En vez de sumar log-verosimilitudes símbolo a símbolo (lo que supone símbolos i.i.d. y hace que la creencia converja con seguridad a la condición equivocada si el modelo está mal especificado; ver Apéndice A), se calcula un vector de **features acumuladas** tras observar $k$ bloques y se ajusta **una regresión logística por cada $k$**:

$$b_k = P(z=1 \mid f_{1:k}),\qquad b_0 = \text{tasa base}$$

Así la creencia queda calibrada directamente para cada longitud de observación, sin supuestos de independencia ni temperatura.

### Features acumuladas
Por cada lag $\ell = 1..8$: parte real e imaginaria de la autocorrelación compleja (`re`, `im`) y de la pseudo-autocorrelación sin conjugar (`pre`, `pim`), más $\log \text{EVM}^2$ como covariable de nivel de ruido. Las sumas son **causales**: el par $(t-\ell, t)$ se asigna al bloque que contiene a $t$.

### 4.1 Extracción

In [ ]:
LAGS = tuple(range(1, 9))
COLS = ["log_evm2"] + [f"{t}{l}" for l in LAGS for t in ("re", "im", "pre", "pim")]
# re/im:   parte real/imaginaria de  E[e_t * conj(e_{t+l})]  (autocorrelación compleja)
# pre/pim: parte real/imaginaria de  E[e_t * e_{t+l}]        (pseudo-autocorrelación)


def cumulative_stats(seg):
    """Estadísticas acumuladas tras k = 1..MAX_BLOCKS bloques. Devuelve (MAX_BLOCKS, len(COLS))."""
    e = error_vector(seg[: BLOCK_SIZE * MAX_BLOCKS])
    cols = [np.abs(e) ** 2]
    for lag in LAGS:
        prod = np.zeros(len(e), dtype=complex)
        prod[lag:] = e[:-lag] * np.conj(e[lag:])
        pseudo = np.zeros(len(e), dtype=complex)
        pseudo[lag:] = e[:-lag] * e[lag:]
        cols += [prod.real, prod.imag, pseudo.real, pseudo.imag]
    per_block = np.stack([c.reshape(MAX_BLOCKS, BLOCK_SIZE).sum(axis=1) for c in cols])
    cum = np.cumsum(per_block, axis=1)
    n_sym = BLOCK_SIZE * np.arange(1, MAX_BLOCKS + 1)
    return np.vstack([np.log(cum[0] / n_sym)[None, :], cum[1:] / cum[0]]).T


def extract_features(df_partition):
    ep_len = BLOCK_SIZE * MAX_BLOCKS
    out = []
    for _, row in df_partition.iterrows():
        y = load_capture(row).y
        for s in range(min(N_SLICES_PER_CAPTURE, len(y) // ep_len)):
            out.append({"capture_id": row["capture_id"], "slice_idx": s,
                        "chsp": float(row["chsp_ghz"]), "osnr": float(row["osnr_db"]),
                        "z": int(row["label_geometrica"]),
                        "X": cumulative_stats(y[s * ep_len:(s + 1) * ep_len])})
    return out


FEATS = {name: extract_features(df) for name, df in PARTITIONS.items()}
print(f"{RUN_TAG}: episodios por partición:", {k: len(v) for k, v in FEATS.items()})

### 4.2 ¿Qué features conviene usar? Comparación de variantes

| Variante | Features |
|---|---|
| A | `log_evm2` + parte real, lags 1, 2, 3, 5 |
| B | `log_evm2` + parte real, lags 1..8 |
| C | B + parte imaginaria (autocorrelación compleja) |
| D | C + pseudo-autocorrelación |

Cada variante se ajusta en song1 y se evalúa en song3, barriendo la regularización. Se reporta la mejor por Brier.

In [ ]:
VARIANTS = {
    "A actual (re 1,2,3,5)": ["log_evm2", "re1", "re2", "re3", "re5"],
    "B re 1..8": ["log_evm2"] + [f"re{l}" for l in LAGS],
    "C complejo 1..8 (re+im)": ["log_evm2"] + [f"{t}{l}" for l in LAGS for t in ("re", "im")],
    "D C + pseudo": COLS,
}
assert FINAL_VARIANT in VARIANTS


def mat(eps, k, names):
    idx = [COLS.index(n) for n in names]
    return np.array([e["X"][k - 1][idx] for e in eps]), np.array([e["z"] for e in eps])


ks_report = sorted({1, MAX_BLOCKS // 2, MAX_BLOCKS})
rows = []
for vname, names in VARIANTS.items():
    for k in ks_report:
        Xtr, ztr = mat(FEATS["post"], k, names)
        Xva, zva = mat(FEATS["val"], k, names)
        for C in (0.03, 0.1, 0.3, 1.0):
            p = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(Xtr, ztr).predict_proba(Xva)[:, 1]
            rows.append({"variante": vname, "k": k, "C": C, "AUC": roc_auc_score(zva, p), "Brier": np.mean((p - zva) ** 2)})
res_variants = pd.DataFrame(rows)
best = res_variants.loc[res_variants.groupby(["variante", "k"])["Brier"].idxmin()]
z_val = np.array([e["z"] for e in FEATS["val"]])
print(f"Mejor regularización por variante (song3). Referencia: Brier de predecir la tasa base = "
      f"{z_val.mean() * (1 - z_val.mean()):.3f}")
display(best.pivot(index="variante", columns="k", values=["AUC", "Brier"]).round(3))

### 4.3 Posterior final y episodios de creencias

Con la variante elegida (`FINAL_VARIANT`) se ajusta un modelo por cada $k$ **solo en song1** y se generan las trayectorias de creencias $b_0, b_1, \dots, b_H$ de cada episodio de las cuatro particiones. Estos episodios son la entrada de las secciones 5 y 6.

In [ ]:
NAMES_FINAL = VARIANTS[FINAL_VARIANT]
IDX_FINAL = [COLS.index(n) for n in NAMES_FINAL]
H = MAX_BLOCKS

MODELS = {
    k: make_pipeline(StandardScaler(), LogisticRegression(C=C_REG, max_iter=3000)).fit(*mat(FEATS["post"], k, NAMES_FINAL))
    for k in range(1, H + 1)
}
PRIOR_P1 = float(np.mean([e["z"] for e in FEATS["post"]]))    # b_0: tasa base de song1


def to_episodes(eps):
    Xall = np.stack([e["X"] for e in eps])[:, :, IDX_FINAL]    # (E, H, n_features)
    P = np.column_stack([np.full(len(eps), PRIOR_P1)] +
                        [MODELS[k].predict_proba(Xall[:, k - 1, :])[:, 1] for k in range(1, H + 1)])
    return [{"capture_id": e["capture_id"], "slice_idx": e["slice_idx"], "z": e["z"], "chsp": e["chsp"],
             "osnr": e["osnr"], "beliefs": P[i], "blocks_available": H, "symbols_per_block": BLOCK_SIZE}
            for i, e in enumerate(eps)]


train_episodes = to_episodes(FEATS["policy"])    # song2
val_episodes = to_episodes(FEATS["val"])         # song3
test_episodes = to_episodes(FEATS["test"])       # song4
print(f"Variante {FINAL_VARIANT} | prior b_0 = {PRIOR_P1:.3f} | episodios: "
      f"políticas={len(train_episodes)}, val={len(val_episodes)}, test={len(test_episodes)}")

### 4.4 Diagnóstico del posterior en validación (song3)

Se compara con los baselines triviales (clase mayoritaria y Brier de predecir la tasa base), y se revisa dónde están los errores, si la creencia está calibrada y si la **confianza crece cuando el modelo se equivoca** (el síntoma del enfoque anterior).

In [ ]:
B = np.array([e["beliefs"] for e in val_episodes])
Z = np.array([e["z"] for e in val_episodes])
CH = np.array([e["chsp"] for e in val_episodes])
OS = np.array([e["osnr"] for e in val_episodes])
base_rate = Z.mean()
print(f"Baselines en val: clase mayoritaria acc = {max(base_rate, 1 - base_rate):.3f} | "
      f"Brier de predecir la tasa base = {base_rate * (1 - base_rate):.3f}")

rows = []
for k in range(1, H + 1):
    b = B[:, k]
    wrong = (b >= 0.5) != (Z == 1)
    rows.append({"k": k, "AUC": roc_auc_score(Z, b), "Brier": np.mean((b - Z) ** 2), "acc": np.mean(~wrong),
                 "conf_media_si_error": np.mean(np.maximum(b, 1 - b)[wrong]) if wrong.any() else np.nan,
                 "%sat(<.01|>.99)": 100 * np.mean((b < .01) | (b > .99))})
diag_k = pd.DataFrame(rows).set_index("k")
show_k = diag_k if H <= 12 else diag_k.loc[[1, 2, 3, 5, 8, 10, 15, 20, 25, H]]
display(show_k.round(3))
diag_k.reset_index().to_csv(PROCESSED_DIR / f"clasificador_{RUN_TAG}.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].plot(diag_k.index, diag_k["AUC"], marker="o")
axes[0].set(title="AUC vs bloques observados", xlabel="k", ylabel="AUC (song3)")
axes[1].plot(diag_k.index, diag_k["Brier"], marker="o", label="posterior")
axes[1].axhline(base_rate * (1 - base_rate), color="gray", ls="--", label="tasa base")
axes[1].set(title="Brier vs bloques observados", xlabel="k", ylabel="Brier (song3)")
axes[1].legend()
bins = np.linspace(0, 1, 11)
cal = pd.DataFrame({"b": B[:, H], "z": Z})
cal["bin"] = pd.cut(cal["b"], bins, include_lowest=True)
g = cal.groupby("bin", observed=True).agg(n=("z", "size"), pred=("b", "mean"), real=("z", "mean"))
axes[2].plot([0, 1], [0, 1], "k--", lw=1)
axes[2].scatter(g["pred"], g["real"], s=20 + 3 * g["n"], alpha=0.7)
axes[2].set(title=f"Confiabilidad tras {H} bloques", xlabel="creencia predicha", ylabel="frecuencia real de z=1")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_dx = pd.DataFrame({"chsp": CH, "osnr": OS, "z": Z, "b": B[:, H]})
df_dx["acierto@0.5"] = (df_dx["b"] >= 0.5) == (df_dx["z"] == 1)
df_dx["declara_traslape"] = df_dx["b"] >= BAYES_THR

print(f"Por ChSp, tras {H} bloques (song3):")
display(df_dx.groupby("chsp").agg(n=("z", "size"), z=("z", "first"), b_media=("b", "mean"),
                                  acierto=("acierto@0.5", "mean"),
                                  declara_traslape=("declara_traslape", "mean")).round(3))

z0_dx = df_dx[df_dx["z"] == 0].copy()
z0_dx["osnr_bin"] = pd.cut(z0_dx["osnr"], [0, 22, 26, 30, 100])
print("\nSolo z=0, por OSNR (¿se declara traslape de más con OSNR bajo?):")
display(z0_dx.groupby("osnr_bin", observed=True).agg(n=("z", "size"), b_media=("b", "mean"),
                                                     declara_traslape=("declara_traslape", "mean")).round(3))

print(f"\nConfiabilidad tras {H} bloques:")
display(g.round(3))

### 4.5 ¿El límite está en el modelo o en las features?

Si un modelo más flexible sobre las **mismas** features no mejora de forma clara, el techo está en la información que contienen las features y no en el clasificador. Se compara sobre la variante A (5 features), la misma con la que se hizo el diagnóstico original.

In [ ]:
cands = {
    "LR (actual)": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000)),
    "LR + cuadráticos": lambda: make_pipeline(StandardScaler(), PolynomialFeatures(2, include_bias=False),
                                              StandardScaler(), LogisticRegression(C=0.3, max_iter=3000)),
    "Gradient boosting": lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                                max_iter=200, random_state=SEED),
}
rows = []
for k in ks_report:
    Xtr, ztr = mat(FEATS["post"], k, VARIANTS["A actual (re 1,2,3,5)"])
    Xva, zva = mat(FEATS["val"], k, VARIANTS["A actual (re 1,2,3,5)"])
    for name, make in cands.items():
        p = make().fit(Xtr, ztr).predict_proba(Xva)[:, 1]
        rows.append({"k": k, "modelo": name, "AUC": roc_auc_score(zva, p), "Brier": np.mean((p - zva) ** 2)})
display(pd.DataFrame(rows).round(3))

---
## 5. Valor de observar

### Formulación como problema de parada
- **Estado:** $(k, b_k)$, número de bloques observados y creencia actual (discretizada en `N_BINS` intervalos para las políticas tabulares).
- **Acciones:** declarar *sin traslape* (0), declarar *traslape* (1), o *observar* otro bloque (2; solo si $k < H$).
- **Costos:** observar cuesta `C_OBS` por bloque; declarar 0 con z = 1 cuesta `C_FN`; declarar 1 con z = 0 cuesta `C_FP`; acertar no cuesta.
- **Umbral de Bayes:** con la creencia calibrada, declarar traslape si $b \ge C_{FP}/(C_{FP}+C_{FN}) = 1/3$.

**Observación importante:** las acciones no alteran los datos. La trayectoria de creencias de un episodio es la misma sin importar lo que se decida, y el resultado de cada declaración se conoce porque se tiene z. Es un problema de parada con datos *fuera de línea*, y por eso no hace falta explorar: el óptimo se podría obtener por programación dinámica sobre las trayectorias (sección 6).

### Costo de decidir tras exactamente N bloques
`costo_decision` no incluye el costo de observar. El costo total es `costo_decision + C_OBS · N`.

In [ ]:
def decision_costs(B, Z, thr=BAYES_THR):
    """Costo medio de decisión (sin observación) si siempre se declara tras N bloques."""
    rows = []
    for k in range(B.shape[1]):
        decl = B[:, k] >= thr
        fn, fp = (Z == 1) & ~decl, (Z == 0) & decl
        rows.append({"N": k, "AUC": roc_auc_score(Z, B[:, k]) if k > 0 else np.nan,
                     "FNR": fn.sum() / (Z == 1).sum(), "FPR": fp.sum() / (Z == 0).sum(),
                     "costo_decision": (C_FN * fn + C_FP * fp).mean()})
    return pd.DataFrame(rows).set_index("N")


dc = decision_costs(B, Z)
dec = dc["costo_decision"].to_numpy()
show_n = dc if H <= 12 else dc.loc[[0, 1, 2, 3, 5, 8, 10, 15, 20, 25, H]]
print(f"Validación (song3). Baseline trivial (N=0, declarar traslape sin observar): costo = {dec[0]:.3f}")
display(show_n.round(3))

# C_OBS crítico: el mayor precio por bloque para el cual observar N>0 bloques aún bate al trivial
gain_per_block = (dec[0] - dec[1:]) / np.arange(1, len(dec))
c_crit, n_crit = gain_per_block.max(), int(gain_per_block.argmax() + 1)
print(f"\nC_OBS crítico ≈ {c_crit:.4f} por bloque (alcanzado con N = {n_crit}). "
      f"Por encima, la mejor política sin más información es no observar.")

grid_c = [0.005, 0.01, 0.02, 0.03, 0.04, 0.05]
rows = []
for c in grid_c:
    tot = dec + c * np.arange(len(dec))
    n = int(tot.argmin())
    rows.append({"C_OBS": c, "mejor_N": n, "costo": tot[n], "trivial": tot[0], "ahorro_%": 100 * (1 - tot[n] / tot[0])})
print("\nMejor ventana fija según el precio de observar (elegida sobre la misma song3: algo optimista):")
display(pd.DataFrame(rows).round(3))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
Ns = np.arange(len(dec))
for c in C_OBS_SWEEP:
    ax.plot(Ns, dec + c * Ns, marker="o", ms=3, label=f"C_OBS = {c}")
ax.axhline(dec[0], color="k", ls=":", label="trivial (N = 0)")
ax.set(title="Costo total de la ventana fija vs N (song3)", xlabel="N (bloques observados)", ylabel="costo medio")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Políticas de parada

Se comparan seis políticas, todas evaluadas en **song4** con las mismas creencias:

| Política | Qué es | Cómo se ajusta |
|---|---|---|
| Trivial | Declarar traslape sin observar | nada |
| Ventana fija | Observar exactamente N bloques y decidir con el umbral de Bayes | N elegido en song3 |
| Umbral | Parar cuando $b \le \eta_{\text{bajo}}$ (declarar 0) o $b \ge \eta_{\text{alto}}$ (declarar 1); al llegar al horizonte, umbral de Bayes | $(\eta_{\text{bajo}}, \eta_{\text{alto}})$ elegidos en song3 |
| SARSA | Control tabular on-policy, $\varepsilon$-greedy | song2, 5 semillas |
| Q-learning | Igual, pero el objetivo usa el máximo del siguiente estado | song2, 5 semillas |
| DP | Iteración de valor hacia atrás sobre las trayectorias de song2, con suavizado hacia el valor esperado por la creencia (pseudo-conteo `m`) | song2; `m` elegido en song3 |

La Q-table se **inicializa** con el costo esperado inmediato (`−C_FN·b`, `−C_FP·(1−b)`); así los empates en estados poco visitados no se resuelven declarando "sin traslape" (la acción más cara). Como `C_OBS` es un parámetro de diseño y no una constante física, se **barre** en lugar de fijarse.

In [ ]:
BS = BLOCK_SIZE
THR = BAYES_THR
ACTS_ALL, ACTS_H = [0, 1, 2], [0, 1]


def bbin(b):
    return min(int(b * N_BINS), N_BINS - 1)


def term_reward(a, z):
    if a == 1:
        return 0.0 if z == 1 else -C_FP
    return 0.0 if z == 0 else -C_FN


def init_q(c):
    q = np.zeros((H + 1, N_BINS, 3))
    centers = (np.arange(N_BINS) + 0.5) / N_BINS
    q[:, :, 0] = -C_FN * centers
    q[:, :, 1] = -C_FP * (1 - centers)
    q[:, :, 2] = np.maximum(q[:, :, 0], q[:, :, 1]) - c
    q[H, :, 2] = -np.inf            # no se puede observar en el horizonte
    return q


def pick(q, k, b, eps, rng):
    acts = ACTS_ALL if k < H else ACTS_H
    if rng.random() < eps:
        return int(rng.choice(acts))
    return int(acts[int(np.argmax(q[k, bbin(b), acts]))])


def train_sarsa(episodes, c, seed):
    rng = np.random.default_rng(seed)
    q = init_q(c)
    for i in range(N_EP):
        eps = EPS0 + (EPS1 - EPS0) * i / (N_EP - 1)
        ep = episodes[int(rng.integers(len(episodes)))]
        b, z = ep["beliefs"], ep["z"]
        k = 0
        a = pick(q, k, b[k], eps, rng)
        while True:
            s = bbin(b[k])
            if a in (0, 1):
                q[k, s, a] += ALPHA * (term_reward(a, z) - q[k, s, a])
                break
            k2 = k + 1
            a2 = pick(q, k2, b[k2], eps, rng)
            q[k, s, 2] += ALPHA * (-c + q[k2, bbin(b[k2]), a2] - q[k, s, 2])
            k, a = k2, a2
    return q


def train_qlearning(episodes, c, seed):
    rng = np.random.default_rng(seed)
    q = init_q(c)
    for i in range(N_EP):
        eps = EPS0 + (EPS1 - EPS0) * i / (N_EP - 1)
        ep = episodes[int(rng.integers(len(episodes)))]
        b, z = ep["beliefs"], ep["z"]
        k = 0
        while True:
            s = bbin(b[k])
            a = pick(q, k, b[k], eps, rng)
            if a in (0, 1):
                q[k, s, a] += ALPHA * (term_reward(a, z) - q[k, s, a])
                break
            k2 = k + 1
            nxt = q[k2, bbin(b[k2]), ACTS_ALL if k2 < H else ACTS_H].max()
            q[k, s, 2] += ALPHA * (-c + nxt - q[k, s, 2])
            k = k2
    return q


def dp_q(episodes, c, m=5.0):
    """Iteración de valor hacia atrás sobre las trayectorias empíricas (sin exploración)."""
    centers = (np.arange(N_BINS) + 0.5) / N_BINS
    S = np.array([[bbin(x) for x in ep["beliefs"]] for ep in episodes])
    Zs = np.array([ep["z"] for ep in episodes])
    q = init_q(c)
    V = np.zeros((H + 1, N_BINS))
    for k in range(H, -1, -1):
        for s in range(N_BINS):
            idx = np.flatnonzero(S[:, k] == s)
            n = len(idx)
            q[k, s, 0] = (np.where(Zs[idx] == 1, -C_FN, 0.0).sum() + m * (-C_FN * centers[s])) / (n + m)
            q[k, s, 1] = (np.where(Zs[idx] == 0, -C_FP, 0.0).sum() + m * (-C_FP * (1 - centers[s]))) / (n + m)
            if k < H:
                cont = V[k + 1][S[idx, k + 1]].sum() if n else 0.0
                q[k, s, 2] = -c + (cont + m * max(q[k, s, 0], q[k, s, 1])) / (n + m)
            V[k, s] = q[k, s, ACTS_ALL if k < H else ACTS_H].max()
    return q


def greedy_policy(q):
    def f(ep):
        b = ep["beliefs"]
        for k in range(H + 1):
            acts = ACTS_ALL if k < H else ACTS_H
            a = acts[int(np.argmax(q[k, bbin(b[k]), acts]))]
            if a in (0, 1):
                return a, k
        return int(b[H] >= THR), H
    return f


def fixed_policy(n):
    return lambda ep: (int(ep["beliefs"][n] >= THR), n)


def thr_policy(lo, hi):
    def f(ep):
        b = ep["beliefs"]
        for k in range(H + 1):
            if b[k] <= lo:
                return 0, k
            if b[k] >= hi:
                return 1, k
        return int(b[H] >= THR), H
    return f


def evaluate(episodes, policy, c):
    r = []
    for ep in episodes:
        a, n = policy(ep)
        z = ep["z"]
        r.append((z, a, n, -term_reward(a, z) + c * n))
    z, a, n, cost = np.array(r, float).T
    return {"costo": cost.mean(), "bloques": n.mean(), "simbolos": n.mean() * BS,
            "FNR": ((z == 1) & (a == 0)).sum() / max(1, (z == 1).sum()),
            "FPR": ((z == 0) & (a == 1)).sum() / max(1, (z == 0).sum())}

In [ ]:
LOS, HIS = (0.05, 0.1, 0.15, 0.2, 0.25, 0.3), (0.5, 0.6, 0.7, 0.8, 0.9, 0.95)
rows = []


def add(c, familia, nombre, metrics, std=np.nan):
    rows.append({"C_OBS": c, "familia": familia, "politica": nombre, **metrics, "costo_std_semillas": std})


def seed_mean(episodes_test, trainer, c):
    res = pd.DataFrame([evaluate(episodes_test, greedy_policy(trainer(train_episodes, c, s)), c) for s in SEEDS])
    return res.mean().to_dict(), res["costo"].std()


for c in C_OBS_SWEEP:
    # Referencias: se ajustan SOLO en validación (song3)
    n_best = min(range(H + 1), key=lambda n: evaluate(val_episodes, fixed_policy(n), c)["costo"])
    lo_hi = min(((lo, hi) for lo in LOS for hi in HIS if lo < hi),
                key=lambda p: evaluate(val_episodes, thr_policy(*p), c)["costo"])
    add(c, "Trivial", "Trivial (decidir ya: traslape)", evaluate(test_episodes, fixed_policy(0), c))
    add(c, "Ventana fija", f"Ventana fija (N={n_best})", evaluate(test_episodes, fixed_policy(n_best), c))
    add(c, "Umbral", f"Umbral {lo_hi}", evaluate(test_episodes, thr_policy(*lo_hi), c))
    # Aprendizaje: song2, 5 semillas
    m_, s_ = seed_mean(test_episodes, train_sarsa, c)
    add(c, "SARSA", "SARSA (media 5 semillas)", m_, s_)
    m_, s_ = seed_mean(test_episodes, train_qlearning, c)
    add(c, "Q-learning", "Q-learning (media 5 semillas)", m_, s_)
    m_best = min((1.0, 5.0, 20.0), key=lambda m: evaluate(val_episodes, greedy_policy(dp_q(train_episodes, c, m)), c)["costo"])
    add(c, "DP", f"DP sobre trayectorias (m={m_best:g})", evaluate(test_episodes, greedy_policy(dp_q(train_episodes, c, m_best)), c))

res = pd.DataFrame(rows)
res.to_csv(PROCESSED_DIR / f"resultados_politicas_{RUN_TAG}.csv", index=False)
print(f"Evaluación en song4 ({RUN_TAG}: H={H} bloques de {BS} símbolos). Referencias y m ajustados en song3.")
display(res.set_index(["C_OBS", "politica"])[["costo", "bloques", "simbolos", "FNR", "FPR", "costo_std_semillas"]].round(3))

In [ ]:
fam_order = ["Trivial", "Ventana fija", "Umbral", "SARSA", "Q-learning", "DP"]
pv_cost = res.pivot(index="C_OBS", columns="familia", values="costo")[fam_order]
pv_sym = res.pivot(index="C_OBS", columns="familia", values="simbolos")[fam_order]
pv_std = res.pivot(index="C_OBS", columns="familia", values="costo_std_semillas")[fam_order]
x = np.arange(len(pv_cost))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for fam in fam_order:
    yerr = pv_std[fam].to_numpy()
    axes[0].errorbar(x, pv_cost[fam], yerr=np.where(np.isnan(yerr), 0, yerr), marker="o", capsize=3, label=fam)
    axes[1].plot(x, pv_sym[fam], marker="o", label=fam)
for ax, title, yl in ((axes[0], "Costo medio en song4 (barras: std entre semillas)", "costo"),
                      (axes[1], "Símbolos observados en promedio", "símbolos")):
    ax.set_xticks(x)
    ax.set_xticklabels(pv_cost.index)
    ax.set(title=title, xlabel="C_OBS (costo por bloque)", ylabel=yl)
    ax.grid(alpha=0.3)
axes[0].legend()
plt.tight_layout()
plt.show()

### 6.1 Curvas de escala: SARSA y Q-learning

Dos preguntas, con el precio de observación `C_SCALE` fijo:

- **Curva de aprendizaje.** Costo en validación y símbolos observados tras *t* actualizaciones (media ± desviación de 5 semillas). Muestra si convergen, si se estabilizan por encima de las referencias y si el paro prematuro de SARSA persiste.
- **Escala de datos.** Con el mismo número de actualizaciones, ¿cuánto baja el costo al entrenar con 10 %, 25 %, 50 % y 100 % de las capturas de song2? Se submuestrea por **captura** (no por ventana), porque las ventanas de una misma captura están correlacionadas. Si las curvas siguen bajando hacia las referencias, el límite son los datos; si se aplanan lejos de ellas, no.

Todo se evalúa en **song3** para no tocar song4. Las referencias (ventana fija y umbral) se ajustaron en song3, así que en este gráfico quedan algo favorecidas.

In [ ]:
C_SCALE = 0.01                                         # precio por bloque con el que se hacen las curvas
CHECKPOINTS = np.unique(np.geomspace(50, N_EP, 14).astype(int))
FRACTIONS = (0.1, 0.25, 0.5, 1.0)
KINDS = ("SARSA", "Q-learning")


def train_td(episodes, c, seed, kind, checkpoints=()):
    # SARSA o Q-learning; con checkpoints, evalúa la política greedy en validación en esos pasos
    rng = np.random.default_rng(seed)
    q = init_q(c)
    cps = {int(x) for x in checkpoints}
    hist = []
    for i in range(1, N_EP + 1):
        eps = EPS0 + (EPS1 - EPS0) * (i - 1) / (N_EP - 1)
        ep = episodes[int(rng.integers(len(episodes)))]
        b, z = ep["beliefs"], ep["z"]
        k = 0
        a = pick(q, k, b[k], eps, rng)
        while True:
            s = bbin(b[k])
            if a in (0, 1):
                q[k, s, a] += ALPHA * (term_reward(a, z) - q[k, s, a])
                break
            k2 = k + 1
            a2 = pick(q, k2, b[k2], eps, rng)
            if kind == "SARSA":
                nxt = q[k2, bbin(b[k2]), a2]
            else:
                nxt = q[k2, bbin(b[k2]), ACTS_ALL if k2 < H else ACTS_H].max()
            q[k, s, 2] += ALPHA * (-c + nxt - q[k, s, 2])
            k, a = k2, a2
        if i in cps:
            m = evaluate(val_episodes, greedy_policy(q), c)
            hist.append({"paso": i, "costo": m["costo"], "simbolos": m["simbolos"]})
    return q, hist


# Referencias en validación (ajustadas en song3, ver nota arriba)
n_best_s = min(range(H + 1), key=lambda n: evaluate(val_episodes, fixed_policy(n), C_SCALE)["costo"])
lo_hi_s = min(((lo, hi) for lo in LOS for hi in HIS if lo < hi),
              key=lambda p: evaluate(val_episodes, thr_policy(*p), C_SCALE)["costo"])
refs_val = {
    "Trivial": evaluate(val_episodes, fixed_policy(0), C_SCALE)["costo"],
    f"Ventana fija (N={n_best_s})": evaluate(val_episodes, fixed_policy(n_best_s), C_SCALE)["costo"],
    f"Umbral {lo_hi_s}": evaluate(val_episodes, thr_policy(*lo_hi_s), C_SCALE)["costo"],
}

# (a) y (b): curva de aprendizaje
lc_rows = []
for kind in KINDS:
    for sd in SEEDS:
        _, hist = train_td(train_episodes, C_SCALE, sd, kind, CHECKPOINTS)
        lc_rows += [{"metodo": kind, "semilla": sd, **h} for h in hist]
lc = pd.DataFrame(lc_rows)

# (c): escala de datos, submuestreando capturas de song2
caps = sorted({e["capture_id"] for e in train_episodes})
sc_rows = []
for f in FRACTIONS:
    for kind in KINDS:
        for sd in SEEDS:
            rng_sub = np.random.default_rng(1000 + sd)
            keep = set(rng_sub.choice(caps, size=max(2, int(round(f * len(caps)))), replace=False).tolist())
            sub = [e for e in train_episodes if e["capture_id"] in keep]
            q, _ = train_td(sub, C_SCALE, sd, kind)
            sc_rows.append({"fraccion": f, "capturas": len(keep), "episodios": len(sub), "metodo": kind,
                            "semilla": sd, **evaluate(val_episodes, greedy_policy(q), C_SCALE)})
sc = pd.DataFrame(sc_rows)

lc.to_csv(PROCESSED_DIR / f"curva_aprendizaje_{RUN_TAG}.csv", index=False)
sc.to_csv(PROCESSED_DIR / f"curva_escala_datos_{RUN_TAG}.csv", index=False)

colors = {"SARSA": "tab:blue", "Q-learning": "tab:orange"}
ref_styles = [("gray", ":"), ("tab:green", "--"), ("tab:red", "-.")]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for kind in KINDS:
    g = lc[lc["metodo"] == kind].groupby("paso")
    for ax, col in ((axes[0], "costo"), (axes[1], "simbolos")):
        mu, sd_ = g[col].mean(), g[col].std().fillna(0)
        ax.plot(mu.index, mu, marker="o", ms=3, color=colors[kind], label=kind)
        ax.fill_between(mu.index, np.maximum(mu - sd_, 0), mu + sd_, alpha=0.2, color=colors[kind])
    gs = sc[sc["metodo"] == kind].groupby("capturas")["costo"]
    mu, sd_ = gs.mean(), gs.std().fillna(0)
    axes[2].errorbar(mu.index, mu, yerr=sd_, marker="o", capsize=3, color=colors[kind], label=kind)
for (name, v), (col, ls) in zip(refs_val.items(), ref_styles):
    axes[0].axhline(v, color=col, ls=ls, label=name)
    axes[2].axhline(v, color=col, ls=ls, label=name)
axes[0].set(title=f"Curva de aprendizaje (C_OBS = {C_SCALE}, song3)", xlabel="actualizaciones (escala log)",
            ylabel="costo medio", xscale="log")
axes[1].set(title="Símbolos observados en promedio", xlabel="actualizaciones (escala log)",
            ylabel="símbolos", xscale="log")
axes[2].set(title="Escala de datos (mismas actualizaciones)", xlabel="capturas de song2 usadas", ylabel="costo medio")
for ax in axes:
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(PROCESSED_DIR / f"curvas_escala_{RUN_TAG}.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Referencias en validación (C_OBS = {C_SCALE}):", {k: round(float(v), 3) for k, v in refs_val.items()})
print("\nEscala de datos (media ± std entre 5 semillas):")
display(sc.groupby(["metodo", "capturas"]).agg(episodios=("episodios", "first"), costo=("costo", "mean"),
                                               costo_std=("costo", "std"), simbolos=("simbolos", "mean"),
                                               FNR=("FNR", "mean"), FPR=("FPR", "mean")).round(3))

---
## 7. Sensibilidad: horizonte y número de ventanas

Cada corrida guarda `clasificador_<tag>.csv` y `resultados_politicas_<tag>.csv` en `data/processed/`. Para llenar esta sección:

1. En la sección 1 cambiar `MAX_BLOCKS` y/o `N_SLICES_PER_CAPTURE` (por ejemplo `MAX_BLOCKS = 30`, o `N_SLICES_PER_CAPTURE = 30`; una captura tiene ≈ 39 500 símbolos, así que `MAX_BLOCKS × 100 × N_SLICES ≤ 39 500`).
2. Ejecutar el notebook completo.
3. Volver a esta celda.

Las preguntas que responde:
- **Horizonte:** ¿el AUC sigue subiendo con más bloques? Es la palanca que ataca el falso positivo.
- **Más ventanas:** si SARSA, Q-learning y DP se acercan al umbral con 6× más episodios, el problema es escasez de datos; si no, el límite son las 99 condiciones. Las ventanas de una misma captura están correlacionadas, así que no equivalen a datos independientes.

In [ ]:
files_res = sorted(PROCESSED_DIR.glob("resultados_politicas_H*_S*.csv"))
files_clf = sorted(PROCESSED_DIR.glob("clasificador_H*_S*.csv"))
print("Configuraciones guardadas:", [f.stem.replace("resultados_politicas_", "") for f in files_res])

if len(files_res) < 2:
    print("\nSolo hay una configuración guardada: repetir la corrida con otro MAX_BLOCKS / N_SLICES_PER_CAPTURE "
          "(ver instrucciones arriba).")
else:
    allres = pd.concat([pd.read_csv(f).assign(config=f.stem.replace("resultados_politicas_", "")) for f in files_res])
    print("\nCosto medio en song4 por configuración:")
    display(allres.pivot_table(index=["C_OBS", "familia"], columns="config", values="costo").round(3))
    print("Símbolos observados en promedio:")
    display(allres.pivot_table(index=["C_OBS", "familia"], columns="config", values="simbolos").round(0))

if len(files_clf) >= 2:
    allclf = pd.concat([pd.read_csv(f).assign(config=f.stem.replace("clasificador_", "")) for f in files_clf])
    ks = [k for k in (1, 5, 10, 15, 20, 30) if k in set(allclf["k"])]
    print("\nAUC en validación (song3) según bloques observados:")
    display(allclf[allclf["k"].isin(ks)].pivot(index="k", columns="config", values="AUC").round(3))

---
## 8. Conclusiones

> Las cifras provienen de la **corrida de referencia** (H = 10 bloques de 100 símbolos, 5 ventanas por captura, `C_FN = 10`, `C_FP = 5`). Si se cambia la configuración, las tablas de las secciones 4 a 7 son la fuente de verdad.

### Hallazgos

1. **El EVM por sí solo no detecta el traslape.** Su AUC es 0.69 con la etiqueta completa y casi todo lo explica el OSNR (correlación −0.92 dentro de z=0).
2. **La autocorrelación del error sí lo detecta y no depende del OSNR.** `acf1` tiene correlación −0.03 con el OSNR, un poder de separación de ≈ 0.87 en la zona detectable (0.71 en la frontera, 0.80 con la etiqueta completa) y es estable entre canciones. La estructura que aparece con traslape fuerte (signos alternados entre lags) es compatible con energía cerca del borde de banda; es una interpretación que no se verificó.
3. **Un posterior logístico sobre features acumuladas funciona donde el KDE fallaba.** Con la variante C, el AUC sube de 0.742 (1 bloque) a 0.897 (10 bloques) y el Brier baja de 0.189 a 0.122, frente a 0.215 de predecir la tasa base. Con la variante A, la confianza media cuando se equivoca *baja* con k (0.678 → 0.644), al revés que en el enfoque anterior.
4. **El límite del clasificador está en la frontera.** Tras 10 bloques, la creencia media es 0.71 (34.0 GHz), 0.56 (34.5) y 0.44 (35.0), contra 0.32–0.38 en z=0. Sigue habiendo falsos positivos en ≈ 60 % de los z=0 (FPR 0.606 en song4), y ese error domina el costo.
5. **Observar solo vale la pena si es barato.** Con las creencias de validación, el precio crítico es `C_OBS` ≈ 0.037 por bloque de 100 símbolos; por encima la mejor política es decidir sin observar. Es un parámetro de diseño: su valor tiene que justificarse con el costo real de adquirir símbolos, y no escogerse para que el resultado se vea bien. Por eso se barre.
6. **Una regla de umbral gana en este problema.** En song4 los costos son 1.267 / 1.301 / 1.355 / 1.495 para `C_OBS` = 0.005 / 0.01 / 0.02 / 0.05, frente a 1.566 del trivial. Con `C_OBS` ≤ 0.02 logra los mismos errores que la ventana fija de 10 bloques (FNR 0.041, FPR 0.606) usando 614–687 símbolos en vez de 1000 (−31 % a −39 %): para cuando la creencia ya es casi segura de traslape, no sigue observando. Con 0.05 la ganancia sobre el trivial (0.07) es pequeña y con un solo split de prueba la trato como indicio.
7. **El aprendizaje por refuerzo tabular no supera al umbral.**
   - SARSA se detiene pronto (≈ 3.4–3.6 bloques) y comete más errores (FPR 0.63–0.70).
   - Q-learning espera más (≈ 4.8–5.0 bloques) y supera a SARSA solo con `C_OBS` ≤ 0.01; con 0.02 y 0.05 queda igual o peor.
   - La DP sobre trayectorias salió peor que ambos (se detiene en 1.9–4.3 bloques).
   - Las explicaciones que considero plausibles **no se probaron**: sesgo on-policy de SARSA (aprende el valor de la política exploratoria y subestima el valor de esperar), escasez de datos (341 estados frente a 99 condiciones independientes) y, en la DP, un suavizado que empuja hacia decidir ya.

### Limitaciones
- Solo hay **99 condiciones independientes**; las 5 ventanas de cada captura están correlacionadas, así que los 495 episodios por partición no son 495 datos independientes.
- Un único split de prueba y sin intervalos de confianza. Diferencias menores a ≈ 0.1 de costo no son concluyentes (la dispersión entre semillas es 0.02–0.08).
- La zona 34.5–35.0 GHz es casi indistinguible de z=0 con 1000 símbolos (acierto de 30 % en 35.0 GHz).
- Las decisiones duras al símbolo más cercano pueden contaminar el vector de error con OSNR bajo; no se evaluó el efecto.
- Los costos `C_FN`, `C_FP` y `C_OBS` son ilustrativos.

### Trabajo pendiente
- Correr la sección 7 con horizonte largo (`MAX_BLOCKS = 30`) y con más ventanas (`N_SLICES_PER_CAPTURE = 30`).
- Intervalos de confianza por bootstrap sobre **capturas** (no sobre ventanas).
- Estudiar por qué la DP empeora y si una discretización más gruesa de la creencia ayuda a las políticas tabulares.
- Evaluar con la frontera de la etiqueta desplazada o con una banda de indiferencia en 34.5–35.0 GHz.

---
## Apéndice A. Resultado negativo: KDE sobre IQ crudo con temperatura

El enfoque inicial construía un banco de KDE 2D (150 centros) por condición (ChSp, OSNR) a partir del IQ crudo de song1, acumulaba log-verosimilitudes símbolo a símbolo, formaba el posterior sobre las condiciones y lo "calibraba" dividiendo la log-verosimilitud por una temperatura. **No se vuelve a ejecutar**; se deja el resumen porque explica por qué se cambió de enfoque.

**Qué falló (cifras de la corrida original, song3/song4):**
- El Brier "calibrado" fue 0.2755 en el diagnóstico y 0.23 en la grilla, **peor que predecir la tasa base** (0.215). La exactitud (58.6 %) quedó por debajo de la clase mayoritaria (68.7 %).
- El mejor resultado de las políticas era el clasificador trivial: ventana fija N=1 con FPR = 1.0 y costo 1.6157 = `0.05 + 5·31/99`, exactamente el costo de declarar siempre traslape. SARSA y el umbral tuvieron FPR ≈ 0.97–0.98.
- La confianza subía aunque el modelo se equivocara.

**Por qué:**
1. El KDE se ajustó sobre IQ crudo, no sobre el EVM, y el IQ 2D codifica sobre todo el nivel de ruido (OSNR), no el traslape.
2. Con un ancho de banda de 0.2 (el ruido tiene RMS ≈ 0.17 y la constelación se espacia 0.63) se emborronaba la poca información que diferenciaba las clases.
3. Con verosimilitud mal especificada y supuesto i.i.d., el log-likelihood crece linealmente con n y el posterior converge a la condición más cercana en KL, que puede tener la etiqueta equivocada. Más datos dan más certeza sobre la respuesta equivocada.
4. La temperatura (T = 50–70) no corrige eso: solo aplana el posterior hacia el prior. Mejoraba el Brier a costa de eliminar la información, no de calibrarla.
5. La tabla de la grilla de hiperparámetros estaba pegada a mano en el código, no calculada.

**Lección:** el posterior de la sección 4 se calibra ajustando contra la etiqueta en datos de entrenamiento (un modelo por k) y se valida contra los baselines triviales, en vez de ajustar una temperatura sobre una verosimilitud que no discrimina.